In [36]:
import re
import pandas as pd
import numpy as np

def converter_expressao_para_regex(expressao_booleana):
    """Conversor automático de expressões (A OU B) E (C OU D) para Regex Lookahead"""
    if not isinstance(expressao_booleana, str) or not expressao_booleana.strip():
        return ""

    # 1. Encontra tudo o que está dentro dos parênteses (...)
    # Isso isola os grupos que estão separados por " E "
    grupos_parenteses = re.findall(r"\((.*?)\)", expressao_booleana)

    # Se a estrutura não tiver parênteses, tenta quebrar direto pelo " E "
    if not grupos_parenteses:
        grupos_parenteses = expressao_booleana.split(" E ")

    lookaheads = []

    for grupo in grupos_parenteses:
        # Quebra os termos internos pelo " OU "
        termos = [termo.strip() for termo in grupo.split(" OU ") if termo.strip()]

        if not termos:
            continue

        # Ordena do maior termo para o menor (evita que 'DP' engula 'Defensoria Pública')
        termos_ordenados = sorted(termos, key=len, reverse=True)

        # Escapa caracteres especiais que possam existir no texto (como pontos ou interrogações)
        termos_escapados = "|".join(
            [re.escape(termo) for termo in termos_ordenados]
        )

        # Monta o Lookahead para este grupo específico garantindo as fronteiras de palavra (\b)
        # O .*? garante que ele ache o termo em qualquer parte do texto longo
        lookahead_grupo = f"(?=.*?\\b(?:{termos_escapados})\\b)"
        lookaheads.append(lookahead_grupo)

    # Une todos os lookaheads. Para a Regex dar MATCH, TODOS os grupos precisam ser verdadeiros (Lógica "E")
    regex_final = "".join(lookaheads)
    return regex_final


In [37]:
import re
import pandas as pd

def aplicar_regex_no_dataframe(df, df_regex):
    """Aplica as expressões regulares de df_regex no df e lista os temas encontrados."""
    # Inicializa a coluna com listas vazias para cada linha
    df = df.copy()
    df.loc[:, "temas_encontrados"] = [[] for _ in range(len(df))]

    for _, row in df_regex.iterrows():
        regex = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]  # Nome exato da coluna do seu CSV

        # Pula se a regex estiver vazia
        if regex=="NAN":
            continue

        # Cria uma máscara booleana para as linhas que dão match
        matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        )
        total_matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        ).sum()
        print("Total de palavras:", total_matches)
        # Para as linhas onde deu Match, adiciona o código do tema na lista
        df.loc[matches, "temas_encontrados"] = df.loc[
            matches, "temas_encontrados"
        ].apply(lambda x: x + [tema_codigo])

    return df

In [38]:
import re
import pandas as pd

def aplicar_regex_flexivel_no_dataframe(df, df_regex, corte_porcentagem=30.0):
    """
    Aplica as expressões regulares de forma flexível, tratando caracteres especiais
    e corrigindo erros de compilação de Regex.
    """
    df = df.copy()
    
    # Inicializa as colunas de resultados
    df["temas_encontrados"] = [[] for _ in range(len(df))]
    df["termos_capturados"] = [[] for _ in range(len(df))]
    
    for _, row in df_regex.iterrows():
        regex_completa = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]

        if regex_completa == "NAN" or pd.isna(regex_completa):
            continue

        # 1. Extração robusta dos blocos de termos internos
        # Captura o conteúdo de dentro dos parênteses dos Lookaheads
        blocos_sujos = re.findall(r"\?=\.\*\?\\b\(([^)]+)\)\\b", regex_completa)
        if not blocos_sujos:
            blocos_sujos = re.findall(r"\(([^)]+)\)", regex_completa)
            
        # Limpa e valida os blocos extraídos
        blocos = []
        for b in blocos_sujos:
            # Remove escapes manuais de parênteses internos que possam quebrar o motor do re.findall
            b_limpo = b.replace(r'\(', '(').replace(r'\)', ')')
            blocos.append(b_limpo)
            
        total_blocos = len(blocos)
        if total_blocos == 0:
            continue

        print(f"Processando Tema {tema_codigo} (Possui {total_blocos} grupos)...")

        # 2. Avalia linha por linha com tratamento de erro integrado
        for idx, texto in df["inteiro_teor_lematizado"].items():
            if pd.isna(texto) or not isinstance(texto, str):
                continue
                
            blocos_com_match = 0
            termos_da_linha = []

            for bloco in blocos:
                try:
                    # Tenta rodar o bloco original
                    matches_bloco = re.findall(bloco, texto, flags=re.IGNORECASE)
                except re.error:
                    # SE o bloco tiver caracteres especiais quebrados (ex: parênteses soltos),
                    # o Python escapará o bloco dinamicamente para tratá-lo como texto puro literal
                    bloco_seguro = "|".join([re.escape(termo.strip().replace('\\', '')) for termo in bloco.split('|')])
                    matches_bloco = re.findall(bloco_seguro, texto, flags=re.IGNORECASE)
                
                if matches_bloco:
                    blocos_com_match += 1
                    # Garante que salve strings (caso retorne tuplas de grupos)
                    termos_limpos = [str(m) for m in matches_bloco if m]
                    termos_da_linha.extend(list(set(termos_limpos)))

            # 3. Cálculo da nota de corte
            porcentagem_sucesso = (blocos_com_match / total_blocos) * 100
            print(f"Possui:", blocos_com_match)
            print(f"Porcentagem de sucesso:", porcentagem_sucesso)
            if porcentagem_sucesso >= corte_porcentagem:
                df.at[idx, "temas_encontrados"].append(f"{tema_codigo} ({porcentagem_sucesso:.1f}%)")
                termos_unicos = list(set([t for t in termos_da_linha if t.strip()]))
                df.at[idx, "termos_capturados"].append(f"{tema_codigo}: {termos_unicos}")

    return df

In [43]:
pd.set_option('display.max_colwidth', 512)  # Shows full cell contents without cutting off text


In [39]:
df_lematizado = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\dataset_enriquecido_10062026_lematizado.parquet")
df_temas = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Temas_STF_com_Regex.csv")

In [40]:
df_lematizado_sample = df_lematizado.sample(frac=0.01, random_state=42)
df_sample = df_lematizado_sample.iloc[[0]]

In [41]:
df_temas["REGEX"].fillna("NAN",inplace=True)
df_lematizado_sample_with_regex = aplicar_regex_flexivel_no_dataframe(df_sample, df_temas)

C:\Users\lfmelo\AppData\Local\Temp\ipykernel_20564\1473219077.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_temas["REGEX"].fillna("NAN",inplace=True)


Processando Tema 1002 (Possui 13 grupos)...
Possui: 5
Porcentagem de sucesso: 38.46153846153847
Processando Tema 1011 (Possui 14 grupos)...
Possui: 0
Porcentagem de sucesso: 0.0
Processando Tema 1016 (Possui 13 grupos)...
Possui: 1
Porcentagem de sucesso: 7.6923076923076925
Processando Tema 106 (Possui 18 grupos)...
Possui: 1
Porcentagem de sucesso: 5.555555555555555
Processando Tema 1075 (Possui 22 grupos)...
Possui: 1
Porcentagem de sucesso: 4.545454545454546
Processando Tema 1079 (Possui 21 grupos)...
Possui: 1
Porcentagem de sucesso: 4.761904761904762
Processando Tema 1102 (Possui 23 grupos)...
Possui: 2
Porcentagem de sucesso: 8.695652173913043
Processando Tema 1132 (Possui 23 grupos)...
Possui: 2
Porcentagem de sucesso: 8.695652173913043
Processando Tema 1153 (Possui 24 grupos)...
Possui: 0
Porcentagem de sucesso: 0.0
Processando Tema 1170 (Possui 22 grupos)...
Possui: 2
Porcentagem de sucesso: 9.090909090909092
Processando Tema 1172 (Possui 21 grupos)...
Possui: 1
Porcentagem de

In [44]:
df_lematizado_sample_with_regex

,TEMA_ORIGEM,TEMA_CODIGO,inteiro_teor,inteiro_teor_lematizado,temas_encontrados,termos_capturados
4013,STJ,1075,"Erro Parser>>>>>inicio<<<<<\nAvenida Jamel Cecílio, nº 2.496, Edifício New Business Style, Sala 42-B, Jardim Goiás, Goiânia/Go, CEP: 74.810-100 1 EXCELENTÍSSIMO (A) SR (A). DR (A). JUÍZ (A) DE DIREITO DO 2º JUIZADO ESPECIAL DA FAZENDA PÚBLICA ESTADUAL DA COMARCA DE GOIÂNIA – ESTADO DE GOIÁS. PROCESSO: 5157240.26.2016.8.09.0051 LIENDERSON MOREIRA DE MOURA, já qualificado nos autos em epígrafe, vem à digna presença de Vossa Excelência, por intermédio de seu procurador, informar e requerer o que segue: Dia...","Erro Parser>>>>>inicio < < < < < \n Avenida Jamel Cecílio , nº 2.496 , Edifício New Business Style , Sala 42-B , Jardim Goiás , Goiânia / Go , CEP : 74. 810-100 1 EXCELENTÍSSIMO ( o ) SR ( A ) . DR ( A ) . JUÍZ ( A ) DE DIREITO DO 2º JUIZADO ESPECIAL de o FAZENDA PÚBLICA ESTADUAL de o COMARCA de GOIÂNIA – ESTADO DE GOIÁS . PROCESSO : 5157240.26.2016.8.09.0051 LIENDERSON MOREIRA DE MOURA , já qualificar em o auto em epígrafe , vir a o digno presença de Vossa Excelência , por intermédio de seu procurador ...",[1002 (38.5%)],"[1002: ['fazenda Pública', 'aparelhamento', 'estado', 'FAZENDA PÚBLICA', 'fundo', 'SUCUMBÊNCIA', 'integra', 'Estado', 'FUNDO', 'HONORÁRIOS ADVOCATÍCIOS', 'Município', 'ESTADO', 'sucumbência', 'honorários', 'eSTADO', 'ente público']]"
